# SQL 实战：用纯 SQL 复刻全部四个分析阶段

> 本 notebook 属于 `sql_and_bi/sql_work` 模块，目的：**集中展示窗口函数 / CTE / 多表 JOIN / PIVOT / 分位数函数**在真实业务分析中的用法。
> `notebooks/01-04` 中的每一个分析，这里都有对应的**纯 SQL 实现**，口径与结论完全一致。

## 文件对照

| 本目录 SQL 文件 | 对应 notebook | 核心 SQL 技术点 |
|---|---|---|
| `01_business_overview.sql` | `notebooks/01_business_understanding.ipynb` | 聚合窗口函数 `SUM(COUNT(*)) OVER ()`、3 表 JOIN、`GROUP BY ... HAVING` |
| `02_funnel_analysis.sql` | `notebooks/02_funnel_analysis.ipynb` | 多层 CTE、`FIRST_VALUE()` / `LAG()` 算漏斗转化率、`PERCENTILE_CONT` 算耗时分位 |
| `03_rfm_segmentation.sql` | `notebooks/03_rfm_analysis.ipynb` | `QUANTILE_CONT` 分位打分（等价复现 pandas `qcut(duplicates='drop')`）、`MEDIAN()` 阈值分群、窗口占比 |
| `04_retention_cohort.sql` | `notebooks/04_retention_analysis.ipynb` | `ROW_NUMBER() OVER (PARTITION BY ...)`、`FIRST_VALUE()` 定留存基准、`PIVOT` 透视留存矩阵、5 表 JOIN |

## 约定与口径

- 每个 `.sql` 文件内部以 `-- QUERY: <编号> <标题>` 注释分段，**每段都是一条可独立执行的 SQL**。
- 数据口径与 `notebooks/` 完全一致：用户级分析用 `customer_unique_id`，GMV/RFM/留存仅统计 `delivered` 订单。
- 数据路径为相对路径 `../../data/raw/`，因此请在 `sql_and_bi/sql_work/` 目录下运行本 notebook。
- 也可以不依赖本 notebook：安装 [DuckDB CLI](https://duckdb.org/docs/installation/) 后，在同一目录下可直接执行任意一个 `.sql` 文件。

In [1]:
import re
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

con = duckdb.connect()

def run_sql_file(path):
    """逐段执行一个 .sql 文件：以 '-- QUERY:' 注释为分段标记，打印每段标题与结果"""
    text = Path(path).read_text(encoding='utf-8')
    sections = re.split(r'(?m)^--\s*QUERY:\s*', text)
    for sec in sections[1:]:
        title, _, sql = sec.partition('\n')
        print(f"\n{'=' * 78}\n▶ {title.strip()}\n{'=' * 78}")
        df = con.sql(sql.strip()).df()
        with pd.option_context('display.max_rows', 80):
            print(df.to_string(index=False))

print("✅ 环境准备完成（DuckDB 直读 CSV，无需入库）")

✅ 环境准备完成（DuckDB 直读 CSV，无需入库）


## 阶段 01：业务理解 → `01_business_overview.sql`

技术点：聚合窗口函数一次算占比、`order_items ⋈ products ⋈ translation` 三表 JOIN、`HAVING` 筛重复购买用户。

**对账锚点**（与 `notebooks/01` 一致）：订单 99,441 笔，97.02% 送达；唯一用户 96,096；复购用户（多个 `customer_id`）2,997 人；平均客单价 R$137.75；Top 品类 bed_bath_table / health_beauty。

In [2]:
run_sql_file('01_business_overview.sql')


▶ 01-1 数据规模总览（9 张表的行数对比）


          table_name  row_cnt
         geolocation  1000163
         order_items   112650
      order_payments   103886
              orders    99441
           customers    99441
       order_reviews    99224
            products    32951
             sellers     3095
category_translation       71

▶ 01-2 订单状态分布（窗口函数一次算出各状态占比）
order_status  order_cnt   pct
   delivered      96478 97.02
     shipped       1107  1.11
    canceled        625  0.63
 unavailable        609  0.61
    invoiced        314  0.32
  processing        301  0.30
     created          5  0.01
    approved          2  0.00

▶ 01-3 月度订单趋势（口径：仅 delivered 订单）


  month  order_cnt
2016-09          1
2016-10        265
2016-12          1
2017-01        750
2017-02       1653
2017-03       2546
2017-04       2303
2017-05       3546
2017-06       3135
2017-07       3872
2017-08       4193
2017-09       4150
2017-10       4478
2017-11       7289
2017-12       5513
2018-01       7069
2018-02       6555
2018-03       7003
2018-04       6798
2018-05       6749
2018-06       6099
2018-07       6159
2018-08       6351

▶ 01-4 品类表现 Top 20（3 表 JOIN：明细 ⋈ 商品 ⋈ 品类翻译）


                category  order_cnt  total_revenue  avg_price  product_cnt
          bed_bath_table       9417     1036988.68      93.30         3029
           health_beauty       8836     1258681.34     130.16         2444
          sports_leisure       7720      988048.97     114.34         2867
   computers_accessories       6689      911954.32     116.51         1639
         furniture_decor       6449      729762.49      87.56         2657
              housewares       5884      632248.66      90.79         2335
           watches_gifts       5624     1205005.68     201.14         1329
               telephony       4199      323667.53      71.21         1134
                    auto       3897      592720.11     139.96         1900
                    toys       3886      483946.60     117.55         1411
              cool_stuff       3632      635290.85     167.36          789
            garden_tools       3518      485256.46     111.63          753
               perfumery 

id_count  user_cnt    pct
       2      2745  91.59
       3       203   6.77
       4        30   1.00
       5         8   0.27
       6         6   0.20
       7         3   0.10
       9         1   0.03
      17         1   0.03
      合计      2997 100.00

▶ 01-7 用户地理分布（按州去重用户数 + 窗口占比）
customer_state  customer_cnt   pct
            SP         40302 41.92
            RJ         12384 12.88
            MG         11259 11.71
            RS          5277  5.49
            PR          4882  5.08
            SC          3534  3.68
            BA          3277  3.41
            DF          2075  2.16
            ES          1964  2.04
            GO          1952  2.03
            PE          1609  1.67
            CE          1313  1.37
            PA           949  0.99
            MT           876  0.91
            MA           726  0.76
            MS           694  0.72
            PB           519  0.54
            PI           482  0.50
            RN           474  0.49
         

payment_type  pay_cnt   pct  avg_value
 credit_card    76795 73.92     163.32
      boleto    19784 19.04     145.03
     voucher     5775  5.56      65.70
  debit_card     1529  1.47     142.57
 not_defined        3  0.00       0.00

▶ 01-9 评价分数分布
 review_score  review_cnt   pct
            1       11424 11.51
            2        3151  3.18
            3        8179  8.24
            4       19142 19.29
            5       57328 57.78

▶ 01-10 平台关键业务指标汇总（标量子查询拼成单行 KPI）


 delivered_orders  unique_customers  avg_order_value  avg_items_per_order
            96478             96096           137.75                 1.14


## 阶段 02：订单转化漏斗 → `02_funnel_analysis.sql`

技术点：多层 CTE（`order_stages → funnel_stats`）+ `FIRST_VALUE()` 算整体转化率 + `LAG()` 算环节转化率；`PERCENTILE_CONT(...) WITHIN GROUP` 算平均/中位/P95 耗时。

**对账锚点**：99,441 → 99,281 → 97,658 → 96,476，整体送达转化 97.02%；耗时 10.4h / 2.8d / 9.3d（运输是最大瓶颈）。

In [3]:
run_sql_file('02_funnel_analysis.sql')


▶ 02-1 订单漏斗（CTE + FIRST_VALUE/LAG 窗口函数）


            stage  order_cnt  overall_rate  step_rate
      1. Purchase    99441.0        100.00        NaN
      2. Approved    99281.0         99.84      99.84
3. Carrier Pickup    97658.0         98.21      98.37
     4. Delivered    96476.0         97.02      98.79

▶ 02-2 各阶段耗时统计（平均 / 中位 / P95，PERCENTILE_CONT）


           stage  avg_hours  median_hours  p95_hours
purchase→approve       10.4           0.3       48.5
 approve→carrier        2.8           1.8        8.1
carrier→customer        9.3           7.1       24.2

▶ 02-3 交叉分析：各状态订单卡在哪个环节（各阶段平均耗时）
order_status  order_cnt  avg_hr_to_approve  avg_days_to_carrier
   delivered      96478               10.3                  2.8
     shipped       1107               11.7                  3.3
    canceled        625               14.7                  3.2
 unavailable        609               24.4                  NaN
    invoiced        314                9.0                  NaN
  processing        301               17.2                  NaN
     created          5                NaN                  NaN
    approved          2               69.7                  NaN

▶ 02-4 按州拆分漏斗（CTE + 多表 JOIN + HAVING 过滤小样本州）


customer_state  total_orders  delivery_rate  carrier_rate
            SP         41746          97.00         98.06
            RJ         12852          96.12         98.64
            MG         11635          97.59         98.39
            RS          5466          97.77         98.64
            PR          5045          97.58         98.33
            SC          3637          97.53         98.51
            BA          3380          96.33         98.49
            DF          2140          97.20         98.74
            ES          2033          98.13         99.16
            GO          2020          96.88         98.56


## 阶段 03：RFM 用户分层 → `03_rfm_segmentation.sql`

技术点：`QUANTILE_CONT` 切点 + `CASE WHEN` 链实现五分位打分，**完整复现 pandas `qcut(duplicates='drop')` 语义**——
本数据集 97% 用户只买 1 次，frequency 的 4 个分位切点全部退化为 1，自动并箱后全体 F 得分 = 1（与 notebook 的打分逐用户一致）。

**对账锚点**：93,358 位用户；Champions 33,316 人（GMV 占 53.04%）、At Risk 21,269 人（占 35.04%）、Loyal 22,966 人（占 7.01%）、Hibernating 15,807 人（占 4.91%）。

In [4]:
run_sql_file('03_rfm_segmentation.sql')


▶ 03-1 RFM 基础表规模与描述统计（用户级聚合）


 user_cnt  avg_recency  median_recency  min_recency  max_recency  avg_frequency  median_frequency  max_frequency  avg_monetary  median_monetary  max_monetary
    93358       287.48           268.0           50          763           1.03               1.0             15        141.62            89.73       13440.0

▶ 03-2 五分位切点（观察 frequency 切点退化：4 个切点全部 = 1）


 r_q20  r_q40  r_q60  r_q80  f_q20  f_q40  f_q60  f_q80  m_q20  m_q40  m_q60  m_q80
 142.0  227.0  318.0  432.0    1.0    1.0    1.0    1.0   39.9   69.9  109.9  179.9

▶ 03-3 RFM 打分（分位数分箱）与各维度得分分布


          dim  score  user_cnt
F (Frequency)      1     93358
 M (Monetary)      1     19337
 M (Monetary)      2     19436
 M (Monetary)      3     17451
 M (Monetary)      4     18632
 M (Monetary)      5     18502
  R (Recency)      1     18605
  R (Recency)      2     18471
  R (Recency)      3     18723
  R (Recency)      4     18757
  R (Recency)      5     18802

▶ 03-4 RFM 分群统计（中位数阈值法 + 窗口函数算人数/GMV 占比）


              segment  user_cnt  avg_recency  avg_frequency  avg_monetary  total_gmv  user_pct  gmv_pct
      Champions（冠军用户）     33316        183.8           1.06        210.49 7012609.77     35.69    53.04
        At Risk（流失风险）     21269        443.6           1.04        217.82 4632864.90     22.78    35.04
Loyal Customers（忠诚用户）     22966        183.6           1.01         40.34  926492.67     24.60     7.01
    Hibernating（休眠用户）     15807        447.0           1.01         41.09  649530.77     16.93     4.91

▶ 03-5 购买频次分布（验证"97% 用户只买 1 次"）


 frequency  user_cnt   pct
         1     90557 97.00
         2      2573  2.76
         3       181  0.19
         4        28  0.03
         5         9  0.01
         6         5  0.01
         7         3  0.00
         9         1  0.00
        15         1  0.00


## 阶段 04：留存 / 复购 → `04_retention_cohort.sql`

技术点：`MIN()` 定首购月 + `DATEDIFF('month')` 算 cohort 偏移；`FIRST_VALUE()` 窗口定留存基准；`PIVOT ... ON ... IN` 透视留存矩阵宽表；`ROW_NUMBER() OVER (PARTITION BY ...)` 定位首单/第二单；5 表 JOIN 算品类复购率。

**对账锚点**：复购率 3.0%（2,801 / 93,358），最高一人买 15 次；首单后 0-7 天内复购占 36.70%（触达节奏 7d/15d/25d 的依据）。

In [5]:
run_sql_file('04_retention_cohort.sql')


▶ 04-1 整体复购率概览


 total_users  repurchase_users  three_plus_users  repurchase_rate  avg_orders_per_user  max_orders
       93358            2801.0             228.0              3.0                 1.03          15

▶ 04-2 Cohort 购买明细（长表：cohort 月 × 购买偏移月 × 用户数）


cohort_month  offset_month  user_cnt
  2016-09-01             0         1
  2016-10-01             0       262
  2016-10-01             6         1
  2016-10-01             9         1
  2016-10-01            11         1
  2016-10-01            13         1
  2016-10-01            15         1
  2016-10-01            17         1
  2016-10-01            19         2
  2016-10-01            20         2
  2016-12-01             0         1
  2016-12-01             1         1
  2017-01-01             0       717
  2017-01-01             1         2
  2017-01-01             2         2
  2017-01-01             3         1
  2017-01-01             4         3
  2017-01-01             5         1
  2017-01-01             6         3
  2017-01-01             7         1
  2017-01-01             8         1
  2017-01-01            10         3
  2017-01-01            11         1
  2017-01-01            12         5
  2017-01-01            13         3
  2017-01-01            14         1
 

cohort_month  offset_month  user_cnt  retention_pct
  2016-10-01             0       262         100.00
  2016-10-01             6         1           0.38
  2016-10-01             9         1           0.38
  2016-10-01            11         1           0.38
  2016-10-01            13         1           0.38
  2016-10-01            15         1           0.38
  2016-10-01            17         1           0.38
  2016-10-01            19         2           0.76
  2016-10-01            20         2           0.76
  2017-01-01             0       717         100.00
  2017-01-01             1         2           0.28
  2017-01-01             2         2           0.28
  2017-01-01             3         1           0.14
  2017-01-01             4         3           0.42
  2017-01-01             5         1           0.14
  2017-01-01             6         3           0.42
  2017-01-01             7         1           0.14
  2017-01-01             8         1           0.14
  2017-01-01

cohort_month     0   1   2   3   4   5   6   7   8   9  10  11
     2016-10 100.0 NaN NaN NaN NaN NaN 0.4 NaN NaN 0.4 NaN 0.4
     2017-01 100.0 0.3 0.3 0.1 0.4 0.1 0.4 0.1 0.1 NaN 0.4 0.1
     2017-02 100.0 0.2 0.3 0.1 0.4 0.1 0.2 0.2 0.1 0.2 0.1 0.3
     2017-03 100.0 0.4 0.4 0.4 0.4 0.2 0.2 0.3 0.3 0.1 0.4 0.1
     2017-04 100.0 0.6 0.2 0.2 0.3 0.3 0.4 0.3 0.3 0.2 0.3 0.1
     2017-05 100.0 0.5 0.5 0.3 0.3 0.3 0.4 0.1 0.3 0.3 0.3 0.3
     2017-06 100.0 0.5 0.4 0.4 0.3 0.4 0.4 0.2 0.1 0.2 0.3 0.4
     2017-07 100.0 0.5 0.3 0.2 0.3 0.2 0.3 0.1 0.2 0.3 0.2 0.3
     2017-08 100.0 0.7 0.3 0.3 0.3 0.5 0.3 0.3 0.1 0.1 0.2 0.2
     2017-09 100.0 0.7 0.5 0.3 0.4 0.2 0.2 0.2 0.3 0.2 0.2 0.1
     2017-10 100.0 0.7 0.3 0.1 0.2 0.2 0.2 0.4 0.3 0.2 0.2 NaN
     2017-11 100.0 0.6 0.4 0.2 0.2 0.2 0.1 0.2 0.1 0.1 NaN NaN
     2017-12 100.0 0.2 0.3 0.3 0.3 0.2 0.2 0.0 0.2 NaN NaN NaN
     2018-01 100.0 0.3 0.4 0.3 0.3 0.2 0.2 0.2 NaN NaN NaN NaN
     2018-02 100.0 0.3 0.4 0.3 0.3 0.2 0.2 NaN NaN NaN 

 offset_month  avg_retention_pct
            0             100.00
            1               0.48
            2               0.34
            3               0.25
            4               0.29
            5               0.23
            6               0.27
            7               0.21
            8               0.21
            9               0.19
           10               0.26
           11               0.23
           12               0.21
           13               0.20
           14               0.15
           15               0.19
           16               0.14
           17               0.28
           19               0.45
           20               0.76

▶ 04-6 复购时间窗口（ROW_NUMBER 定位首单/第二单 + 间隔分桶）


window_group  user_cnt   pct  avg_days
    0-7 days      1028 36.70       0.7
   8-14 days       152  5.43      10.8
  15-30 days       236  8.43      21.8
  31-60 days       296 10.57      44.6
  61-90 days       196  7.00      74.3
 91-180 days       413 14.74     132.4
   180+ days       480 17.14     286.4

▶ 04-7 品类复购率 Top 15（5 表 JOIN + 条件去重计数）


                category  total_buyers  repurchase_buyers  repurchase_rate
         home_appliances           688                 50             7.27
fashion_bags_accessories          1757                 52             2.96
          bed_bath_table          9008                245             2.72
          sports_leisure          7341                170             2.32
         furniture_decor          6178                126             2.04
   computers_accessories          6405                119             1.86
           health_beauty          8498                142             1.67
               perfumery          3050                 35             1.15
                pet_shop          1667                 19             1.14
           watches_gifts          5421                 61             1.13
              housewares          5681                 60             1.06
                    auto          3769                 40             1.06
               telephony 

## 总结

### 本模块覆盖的 SQL 能力

| 能力 | 具体用法 | 出处 |
|---|---|---|
| **窗口函数** | `SUM(COUNT(*)) OVER ()` 占比、`FIRST_VALUE()` / `LAG()` 漏斗、`ROW_NUMBER() OVER (PARTITION BY ...)` 排序定位、`FIRST_VALUE() OVER (PARTITION BY cohort ORDER BY offset)` 留存基准 | 01 / 02 / 04 |
| **CTE** | 多层 CTE 链（最多 8 层：`reference_date → customer_orders → rfm_base → cuts → scored → thresholds → segmented → 输出`），每段查询独立自洽 | 全部 |
| **多表 JOIN** | 3 表（明细⋈商品⋈翻译）、5 表（`customers ⋈ orders ⋈ order_items ⋈ products ⋈ translation`）、CTE 自 JOIN（首单×第二单） | 01 / 04 |
| **分位数** | `PERCENTILE_CONT(...) WITHIN GROUP` 耗时分位、`QUANTILE_CONT` 分箱切点、`MEDIAN()` 分群阈值 | 02 / 03 |
| **透视** | `PIVOT ... ON offset_month IN (...) USING MAX(...) GROUP BY cohort` 留存矩阵 | 04 |
| **其他** | `GROUP BY ... HAVING`、条件聚合 `COUNT(DISTINCT CASE WHEN ...)`、标量子查询拼 KPI 单行 | 01 / 04 |

### 数字一致性

四个阶段的输出均与 `notebooks/01-04` 逐项对账一致（关键锚点见各阶段标题下的说明）；
其中 03 的 RFM 打分经过 **93,358 用户逐人比对**，SQL 与 pandas `qcut` 结果零差异。

> 下一步：`../BI_DashBoards/` 将本阶段的全部分析产出整合为一个可交互的 BI 看板。